# ✅ NumPy Practice Exercise 2: Complete Solutions & Explanations

Here are the complete, step-by-step solutions and explanations for the **15 advanced machine learning problems** in `04_numpy_exercise_2.ipynb`.


### Problem 1: Vectorized Pairwise Euclidean Distance Matrix

**Question:** Given two matrices $X$ of shape $(N, D)$ and $Y$ of shape $(M, D)$, compute the full pairwise distance matrix of shape $(N, M)$ without any Python loops using broadcasting.

Test with:
- $X$: 4 points in 3D
- $Y$: 5 points in 3D


In [1]:
import numpy as np

def pairwise_distances(X, Y):
    # X: (N, 1, D), Y: (1, M, D) -> diff: (N, M, D)
    diff = X[:, np.newaxis, :] - Y[np.newaxis, :, :]
    return np.sqrt(np.sum(diff ** 2, axis=-1))

rng = np.random.default_rng(42)
X = rng.uniform(0, 10, size=(4, 3))
Y = rng.uniform(0, 10, size=(5, 3))

dist_mat = pairwise_distances(X, Y)
print("Distance Matrix shape:", dist_mat.shape)
print("Distance Matrix (4x5):\n", np.round(dist_mat, 2))


Distance Matrix shape: (4, 5)
Distance Matrix (4x5):
 [[ 5.8   9.72  2.24  6.78  4.62]
 [ 9.04 11.24  5.94  9.45  5.25]
 [ 3.38  5.85  6.52  8.86  6.82]
 [ 6.89  9.1   4.89  6.08  5.92]]


### Problem 2: Numerically Stable Softmax Function

**Question:** Write a function `softmax(logits, axis=-1)` that computes the softmax probabilities across any specified axis.
Make sure it is **numerically stable** against large numbers (e.g. logits like 1000.0) by subtracting the maximum logit before computing exponents.


In [2]:
def softmax(logits, axis=-1):
    # Subtract max along axis for numerical stability
    shifted = logits - np.max(logits, axis=axis, keepdims=True)
    exp_vals = np.exp(shifted)
    return exp_vals / np.sum(exp_vals, axis=axis, keepdims=True)

# Test with potentially overflowing numbers
logits = np.array([[1000.0, 1001.0, 999.0],
                   [1.0, 2.0, 3.0]])
probs = softmax(logits, axis=-1)
print("Softmax Probabilities:\n", np.round(probs, 4))
print("Sum of probabilities along axis=-1:", np.sum(probs, axis=-1))


Softmax Probabilities:
 [[0.2447 0.6652 0.09  ]
 [0.09   0.2447 0.6652]]
Sum of probabilities along axis=-1: [1. 1.]


### Problem 3: Vectorized One-Hot Encoding

**Question:** Write a function `one_hot_encode(labels, num_classes)` that converts a 1D vector of integer class labels into a 2D binary indicator matrix of shape $(N, num\_classes)$ using NumPy indexing.


In [3]:
def one_hot_encode(labels, num_classes):
    return np.eye(num_classes)[labels]

labels = np.array([2, 0, 1, 3, 2, 1])
encoded = one_hot_encode(labels, num_classes=4)
print("Original labels:", labels)
print("One-hot encoded (shape):", encoded.shape)
print("One-hot matrix:\n", encoded)


Original labels: [2 0 1 3 2 1]
One-hot encoded (shape): (6, 4)
One-hot matrix:
 [[0. 0. 1. 0.]
 [1. 0. 0. 0.]
 [0. 1. 0. 0.]
 [0. 0. 0. 1.]
 [0. 0. 1. 0.]
 [0. 1. 0. 0.]]


### Problem 4: Z-Score Feature Standardization with `keepdims=True`

**Question:** Write a function `standardize(X)` that takes a 2D dataset of shape $(N, D)$ and centers each column to mean 0 and scales each column to standard deviation 1. Use `keepdims=True` and guard against division by zero (if std is 0).


In [4]:
def standardize(X, eps=1e-8):
    mean = np.mean(X, axis=0, keepdims=True)
    std = np.std(X, axis=0, keepdims=True)
    return (X - mean) / (std + eps)

rng = np.random.default_rng(10)
data = rng.normal(loc=[50, 100, 2], scale=[10, 25, 0.5], size=(100, 3))

scaled_data = standardize(data)
print("Original mean:", np.round(np.mean(data, axis=0), 2))
print("Original std: ", np.round(np.std(data, axis=0), 2))
print("\nScaled mean (should be ~0):", np.round(np.mean(scaled_data, axis=0), 5))
print("Scaled std  (should be ~1):", np.round(np.std(scaled_data, axis=0), 5))


Original mean: [49.18 93.62  1.87]
Original std:  [ 9.71 19.99  0.51]

Scaled mean (should be ~0): [ 0.  0. -0.]
Scaled std  (should be ~1): [1. 1. 1.]


### Problem 5: Linear Regression via the Normal Equation

**Question:** Given feature matrix $X$ and target vector $y$, find the optimal weight vector $\theta = [\theta_0, \theta_1, \dots]^T$ using the closed-form Normal Equation:
$$\theta = (X_{b}^T X_b)^{-1} X_b^T y$$
Where $X_b$ includes an initial bias column of ones.


In [5]:
rng = np.random.default_rng(42)
N = 100
X = rng.uniform(0, 10, size=(N, 2))
true_weights = np.array([4.2, -1.8])
true_bias = 5.5
y = X @ true_weights + true_bias + rng.normal(0, 0.2, size=N)

# Add bias column
X_b = np.hstack([np.ones((N, 1)), X])

# Solve normal equation
theta = np.linalg.inv(X_b.T @ X_b) @ X_b.T @ y

print(f"Learned Bias     (True: {true_bias}): {theta[0]:.4f}")
print(f"Learned Weights  (True: {true_weights}): {theta[1]:.4f}, {theta[2]:.4f}")


Learned Bias     (True: 5.5): 5.4397
Learned Weights  (True: [ 4.2 -1.8]): 4.2010, -1.7911


### Problem 6: 1D Moving Average Filter via Convolution

**Question:** Given a noisy 1D signal, compute its moving average with a window size of $w=5$ using `np.convolve` in `'valid'` mode.


In [6]:
signal = np.array([1, 3, 2, 5, 8, 7, 6, 9, 10, 8, 5, 3])
window_size = 5
kernel = np.ones(window_size) / window_size

moving_avg = np.convolve(signal, kernel, mode='valid')
print("Original signal length:", len(signal))
print("Moving average length: ", len(moving_avg))
print("Moving average result:\n", np.round(moving_avg, 2))


Original signal length: 12
Moving average length:  8
Moving average result:
 [3.8 5.  5.6 7.  8.  8.  7.6 7. ]


### Problem 7: Find Nearest Neighbor (k-NN core step)

**Question:** Given a dataset of 50 2D points and a query point $q = [2.5, 3.5]$, find:
1. The index of the nearest point.
2. The coordinates of that nearest point.
3. The exact Euclidean distance to that point.


In [7]:
rng = np.random.default_rng(99)
points = rng.uniform(0, 10, size=(50, 2))
query = np.array([2.5, 3.5])

# Vectorized Euclidean distance from query to all points
dists = np.linalg.norm(points - query, axis=1)

nearest_idx = np.argmin(dists)
print(f"Nearest point index: {nearest_idx}")
print(f"Coordinates:         {points[nearest_idx]}")
print(f"Euclidean distance:  {dists[nearest_idx]:.4f}")


Nearest point index: 7
Coordinates:         [2.70271732 3.50014949]
Euclidean distance:  0.2027


### Problem 8: Binary Cross-Entropy Loss with Clipping

**Question:** Write a function `binary_cross_entropy(y_true, y_pred, eps=1e-15)` that calculates binary cross-entropy loss:
$$L = -\frac{1}{N} \sum [y \log(p) + (1-y) \log(1 - p)]$$
Clip `y_pred` to `[eps, 1 - eps]` using `np.clip` to prevent `log(0)` returning `NaN` or `-inf`.


In [8]:
def binary_cross_entropy(y_true, y_pred, eps=1e-15):
    p = np.clip(y_pred, eps, 1.0 - eps)
    loss = -np.mean(y_true * np.log(p) + (1 - y_true) * np.log(1 - p))
    return loss

y_true = np.array([1, 0, 1, 1, 0])
y_pred = np.array([0.99, 0.05, 0.85, 0.70, 0.01])
print("BCE Loss:", round(binary_cross_entropy(y_true, y_pred), 4))

# Extreme edge case test (predictions equal to exactly 0 or 1)
y_edge_pred = np.array([1.0, 0.0, 1.0, 1.0, 0.0])
print("BCE Loss on extreme values (safe with clipping):", round(binary_cross_entropy(y_true, y_edge_pred), 4))


BCE Loss: 0.1181
BCE Loss on extreme values (safe with clipping): 0.0


### Problem 9: Efficient Top-K Extraction with `argpartition`

**Question:** Given an array of 10,000 random floating numbers, find the indices and values of the **top 5 largest numbers** in $O(N)$ expected time using `np.argpartition` without performing a full $O(N \log N)$ sort.


In [9]:
rng = np.random.default_rng(42)
arr = rng.random(10_000)

k = 5
# argpartition puts the k largest elements at the end
top_k_unsorted_idx = np.argpartition(arr, -k)[-k:]

# Sort only the top k elements
top_k_sorted_idx = top_k_unsorted_idx[np.argsort(arr[top_k_unsorted_idx])[::-1]]

print("Top 5 Indices:", top_k_sorted_idx)
print("Top 5 Values: ", arr[top_k_sorted_idx])


Top 5 Indices: [7491 7398 2210 4889 6594]
Top 5 Values:  [0.99997702 0.99986085 0.99976534 0.99971485 0.99951811]


### Problem 10: Image Tensor Channel Transposition & Grayscale Conversion

**Question:** Simulate a batch of 8 RGB images of size 64x64 with shape `(B, H, W, C) = (8, 64, 64, 3)` with random pixel intensities between 0 and 255.
1. Transpose the tensor into PyTorch format `(B, C, H, W)`.
2. Convert the original RGB batch to Grayscale with shape `(8, 64, 64)` using the standard luminance formula: $Y = 0.299 R + 0.587 G + 0.114 B$ using matrix dot product or broadcasting.


In [10]:
rng = np.random.default_rng(7)
images = rng.integers(0, 256, size=(8, 64, 64, 3), dtype=np.uint8)
print("Original batch shape (B, H, W, C):", images.shape)

# 1. Transpose to PyTorch (B, C, H, W)
pytorch_format = np.transpose(images, (0, 3, 1, 2))
print("PyTorch batch shape (B, C, H, W): ", pytorch_format.shape)

# 2. RGB to Grayscale
weights = np.array([0.299, 0.587, 0.114]) # shape (3,)
grayscale = np.dot(images.astype(np.float32), weights)
print("Grayscale batch shape (B, H, W):    ", grayscale.shape)


Original batch shape (B, H, W, C): (8, 64, 64, 3)
PyTorch batch shape (B, C, H, W):  (8, 3, 64, 64)
Grayscale batch shape (B, H, W):     (8, 64, 64)


### Problem 11: Vectorized Cosine Similarity Matrix

**Question:** Given two matrices $A$ of shape $(N, D)$ and $B$ of shape $(M, D)$ containing embeddings, compute the full $N \times M$ pairwise cosine similarity matrix without any Python loops:
$$\text{cosine\_sim}(u, v) = \frac{u \cdot v}{\|u\|_2 \|v\|_2}$$


In [11]:
def cosine_similarity_matrix(A, B, eps=1e-8):
    dot_product = A @ B.T # Shape (N, M)
    norm_A = np.linalg.norm(A, axis=1, keepdims=True) # (N, 1)
    norm_B = np.linalg.norm(B, axis=1, keepdims=True) # (M, 1)
    return dot_product / (norm_A @ norm_B.T + eps)

rng = np.random.default_rng(123)
A = rng.standard_normal((3, 4))
B = rng.standard_normal((5, 4))

sim = cosine_similarity_matrix(A, B)
print("Cosine Similarity Matrix shape:", sim.shape)
print("Similarity values (bounded between -1 and 1):\n", np.round(sim, 3))


Cosine Similarity Matrix shape: (3, 5)
Similarity values (bounded between -1 and 1):
 [[ 0.134 -0.555  0.803  0.378 -0.487]
 [ 0.063  0.595 -0.463  0.149 -0.039]
 [-0.1   -0.296 -0.235 -0.581  0.803]]


### Problem 12: Polynomial Feature Expansion

**Question:** Given a 1D input array $x$ of length $N$, generate the polynomial feature matrix up to degree $d=3$:
$$[1, x, x^2, x^3]$$
Do this two ways:
1. Using column stacking with exponentiation.
2. Using `np.vander(x, N=4, increasing=True)`.


In [12]:
x = np.array([2.0, 3.0, 4.0, 5.0])

# Method 1: Column stack
poly1 = np.column_stack([x**0, x**1, x**2, x**3])

# Method 2: Vandermonde matrix
poly2 = np.vander(x, N=4, increasing=True)

print("Method 1:\n", poly1)
print("Method 2:\n", poly2)
print("Are both methods equal?", np.allclose(poly1, poly2))


Method 1:
 [[  1.   2.   4.   8.]
 [  1.   3.   9.  27.]
 [  1.   4.  16.  64.]
 [  1.   5.  25. 125.]]
Method 2:
 [[  1.   2.   4.   8.]
 [  1.   3.   9.  27.]
 [  1.   4.  16.  64.]
 [  1.   5.  25. 125.]]
Are both methods equal? True


### Problem 13: Principal Component Analysis (PCA) From Scratch

**Question:** Perform PCA on an $N \times 4$ dataset from scratch:
1. Center the features to zero mean.
2. Compute the sample covariance matrix: $\Sigma = \frac{1}{N-1} X_c^T X_c$.
3. Compute eigenvalues and eigenvectors using `np.linalg.eigh`.
4. Sort the eigenvectors in descending order of eigenvalues.
5. Project the data onto the first 2 principal components.


In [13]:
rng = np.random.default_rng(42)
X = rng.normal(loc=[10, 20, 30, 40], scale=[1, 5, 2, 8], size=(100, 4))

# 1. Center
X_c = X - np.mean(X, axis=0)

# 2. Covariance matrix
N = X.shape[0]
cov_matrix = (X_c.T @ X_c) / (N - 1)

# 3. Eigendecomposition
eigenvalues, eigenvectors = np.linalg.eigh(cov_matrix)

# 4. Sort descending
idx = np.argsort(eigenvalues)[::-1]
eigenvalues = eigenvalues[idx]
eigenvectors = eigenvectors[:, idx]

# 5. Project onto top 2 components
top_2_eigenvectors = eigenvectors[:, :2]
X_projected = X_c @ top_2_eigenvectors

print("Eigenvalues (variance explained):", np.round(eigenvalues, 2))
print("Projected data shape: (100, 4) ->", X_projected.shape)


Eigenvalues (variance explained): [50.1  22.58  3.8   0.94]
Projected data shape: (100, 4) -> (100, 2)


### Problem 14: Mini-Batch Generator for Stochastic Gradient Descent

**Question:** Write a Python generator function `get_batches(X, y, batch_size, shuffle=True)` that yields batches $(X_{batch}, y_{batch})$ of size `batch_size`. Ensure the indices are randomly permuted without copying the entire dataset up front.


In [14]:
def get_batches(X, y, batch_size=32, shuffle=True, seed=None):
    N = len(X)
    indices = np.arange(N)
    if shuffle:
        rng = np.random.default_rng(seed)
        indices = rng.permutation(N)
    
    for start_idx in range(0, N, batch_size):
        batch_idx = indices[start_idx : start_idx + batch_size]
        yield X[batch_idx], y[batch_idx]

# Test
X = np.arange(100).reshape(50, 2)
y = np.arange(50)

batch_count = 0
for X_b, y_b in get_batches(X, y, batch_size=16, shuffle=True, seed=42):
    batch_count += 1
    print(f"Batch {batch_count}: X shape {X_b.shape}, y shape {y_b.shape}")


Batch 1: X shape (16, 2), y shape (16,)
Batch 2: X shape (16, 2), y shape (16,)
Batch 3: X shape (16, 2), y shape (16,)
Batch 4: X shape (2, 2), y shape (2,)


### Problem 15: 2D Image Edge Detection (Convolution)

**Question:** Given an $8 \times 8$ grayscale image and a $3 \times 3$ Laplacian edge-detection kernel:
$$K = \begin{pmatrix} 0 & 1 & 0 \\ 1 & -4 & 1 \\ 0 & 1 & 0 \end{pmatrix}$$
Compute the $6 \times 6$ valid convolution output without external libraries using vectorization or a sliding window.


In [15]:
image = np.array([
    [10, 10, 10, 10, 50, 50, 50, 50],
    [10, 10, 10, 10, 50, 50, 50, 50],
    [10, 10, 10, 10, 50, 50, 50, 50],
    [10, 10, 10, 10, 50, 50, 50, 50],
    [10, 10, 10, 10, 50, 50, 50, 50],
    [10, 10, 10, 10, 50, 50, 50, 50],
    [10, 10, 10, 10, 50, 50, 50, 50],
    [10, 10, 10, 10, 50, 50, 50, 50]
], dtype=float)

kernel = np.array([
    [0,  1, 0],
    [1, -4, 1],
    [0,  1, 0]
], dtype=float)

H, W = image.shape
kH, kW = kernel.shape
out_H, out_W = H - kH + 1, W - kW + 1

output = np.zeros((out_H, out_W))

# Convolve sliding window
for r in range(out_H):
    for c in range(out_W):
        window = image[r:r+kH, c:c+kW]
        output[r, c] = np.sum(window * kernel)

print("Original image shape:", image.shape)
print("Convolved edge map:\n", output)


Original image shape: (8, 8)
Convolved edge map:
 [[  0.   0.  40. -40.   0.   0.]
 [  0.   0.  40. -40.   0.   0.]
 [  0.   0.  40. -40.   0.   0.]
 [  0.   0.  40. -40.   0.   0.]
 [  0.   0.  40. -40.   0.   0.]
 [  0.   0.  40. -40.   0.   0.]]
